# Scrapping the ASI Conference Website
**Note:** For reproducibility:



1.   First make a copy of the notebook and work on the copy
2.   Check these things: paths name, file names, APIs IDs, and customize them according to your environment
3.   After doing a major data operation, you can push your final version here: https://github.com/GaelKBertrand/scholar_info_append






### Mounting Google Drive as a working director

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Installing and loading all dependencies and libraries needed

In [2]:
#Install necessary dependencies and libraries
!pip install waybackpy
!pip install googlesearch-python
!pip install google-api-python-client
!pip install beautifulsoup4
!pip install fake_useragent #necessary for google search rapid automation
!pip install selenium
!apt-get update
!apt install -y chromium-chromedriver
!apt-get install -y wget
!pip install requests-html

import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import logging


from waybackpy import WaybackMachineSaveAPI
from googlesearch import search
from googleapiclient.discovery import build
import time
import requests
from fake_useragent import UserAgent
from bs4 import BeautifulSoup
import logging

#from requests_html import HTMLSession
#from requests_html import AsyncHTMLSession
import pandas as pd
import logging
import asyncio
import nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.7/481.7 kB 29.0 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,535 kB]
Get:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy In

## Save dataset

In [ ]:
# Save the cleaned and updated DataFrame to a new CSV
cleaned_df.to_csv("/content/drive/My Drive/ISI2022.csv", index=False)

# Adding Countries

In [3]:
!pip install pandas geopy openpyxl tqdm


In [7]:
!ls /content/drive/MyDrive/ISI/ISIclean/

'Copy of ISI2018r.xlsx'   ISI2018.xlsx	 ISI2021.xlsx   ISI2023.xlsx
 ISI2018r.xlsx		  ISI2019.xlsx	 ISI2022.xlsx


In [8]:
# Load Excel file
file_path = '/content/drive/MyDrive/ISI/ISIclean/ISI2018r.xlsx'  # Replace with the path to your file
xls = pd.ExcelFile(file_path)

In [9]:
import pandas as pd
# List all sheets and their columns
for sheet_name in xls.sheet_names:
    df = pd.read_excel(xls, sheet_name=sheet_name)
    print(f"Sheet: {sheet_name}")
    print(f"Columns: {df.columns.tolist()}")
    print("-" * 40)  # Separator for better readability


Sheet: Authorwise 2018
Columns: ['Session', 'Topic', 'Title of Paper', 'Number of Authors', 'Author', 'Institute/Affiliation', 'Affiliation Country']
----------------------------------------
Sheet: Chair of regular sessions
Columns: ['Conference Year', 'Name of Chair', 'Affiliation', 'Session Theme', 'Affiliation Country']
----------------------------------------
Sheet: SPECIAL sessions Keynote, Plena
Columns: ['Session Type', 'Session Number', 'Presenter/Panelist/Moderator', 'Affiliation', 'Topic', 'Affiliation Country']
----------------------------------------


In [11]:
import pandas as pd

# Sheets to check
required_sheets = ["Authorwise 2018", "SPECIAL sessions Keynote, Plena", "Chair of regular sessions"]
required_columns = ["Institute/Affiliation", "Affiliation Country"]

# Check for sheet existence
missing_sheets = [sheet for sheet in required_sheets if sheet not in xls.sheet_names]
if missing_sheets:
    print(f"Missing sheets: {missing_sheets}")
else:
    print("All required sheets exist.")

# Check for column existence in each sheet
for sheet_name in xls.sheet_names:
    df = pd.read_excel(xls, sheet_name=sheet_name)
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        print(f"Sheet '{sheet_name}' is missing columns: {missing_columns}")
    else:
        print(f"Sheet '{sheet_name}' has all required columns.")


All required sheets exist.
Sheet 'Authorwise 2018' has all required columns.
Sheet 'Chair of regular sessions' is missing columns: ['Institute/Affiliation']
Sheet 'SPECIAL sessions Keynote, Plena' is missing columns: ['Institute/Affiliation']


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import openpyxl

# Initialize geolocator with a rate limiter
geolocator = Nominatim(user_agent="geo_affiliation_locator")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

def get_country_from_affiliation(affiliation):
    """
    Get the country name from the affiliation string using geopy, ensuring results are in English.
    """
    try:
        if pd.isnull(affiliation) or affiliation.strip() == "":
            return None
        location = geocode(affiliation, timeout=10)
        if location and location.raw.get('display_name'):
            address_parts = location.raw['display_name'].split(",")
            # Extract and clean the country name
            country = address_parts[-1].strip()
            return country
        return None
    except Exception as e:
        return None

# Get all sheets
all_sheets = {sheet_name: pd.read_excel(xls, sheet_name=sheet_name) for sheet_name in xls.sheet_names}

# Sheets to process
sheets_to_process = ["Authorwise (2021)", "SPECIAL sessions Keynote, Plena", "Chair of regular sessions"]

# Process each specified sheet
for sheet in sheets_to_process:
    print(f"Processing sheet: {sheet}")
    df = all_sheets[sheet]

    # Identify the relevant columns
    affiliation_col = "Affiliation" if "Affiliation" in df.columns else "Institute/Affiliation"
    country_col = "Affiliation Country"

    # Add the country column if it doesn't exist
    if country_col not in df.columns:
        df[country_col] = None

    # Fill the 'Affiliation Country' column
    tqdm.pandas(desc=f"Processing {sheet}")
    df[country_col] = df[affiliation_col].progress_apply(get_country_from_affiliation)

    # Update the sheet in the dictionary
    all_sheets[sheet] = df

# Save back to Excel, preserving column widths
output_path = "ISI_updated1.xlsx"
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, sheet_df in all_sheets.items():
        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
        # Retain column widths
        workbook = writer.book
        worksheet = writer.sheets[sheet_name]
        for col_num, column_cells in enumerate(sheet_df.columns, start=1):
            max_length = max(sheet_df[column_cells].astype(str).map(len).max(), len(column_cells))
            worksheet.column_dimensions[openpyxl.utils.get_column_letter(col_num)].width = max_length

print(f"Processing complete. Updated file saved as {output_path}.")


Processing sheet: Authorwise (2021)


Processing Authorwise (2021): 100%|██████████| 463/463 [07:43<00:00,  1.00s/it]


Processing sheet: SPECIAL sessions Keynote, Plena


Processing SPECIAL sessions Keynote, Plena: 100%|██████████| 3/3 [00:02<00:00,  1.01it/s]


Processing sheet: Chair of regular sessions


Processing Chair of regular sessions: 100%|██████████| 77/77 [01:17<00:00,  1.00s/it]

Processing complete. Updated file saved as ISI_updated1.xlsx.


Determine location in India or not

In [12]:
import pandas as pd
import openpyxl

# Load Excel file
file_path = "/content/drive/MyDrive/ISI/ISIclean/ISI2018r.xlsx"  # Replace with the path to your file
xls = pd.ExcelFile(file_path)

# Get all sheets
all_sheets = {sheet_name: pd.read_excel(xls, sheet_name=sheet_name) for sheet_name in xls.sheet_names}

# Process all sheets
for sheet in all_sheets.keys():
    print(f"Processing sheet: {sheet}")
    df = all_sheets[sheet]

    # Check if the 'Affiliation Country' column exists
    if "Affiliation Country" not in df.columns:
        print(f"Error: 'Affiliation Country' column not found in {sheet}. Skipping.")
        continue

    # Create the new column 'Affiliation Location'
    df['Affiliation Location'] = df['Affiliation Country'].apply(lambda x: 1 if x == "India" else 0)

    # Update the sheet in the dictionary
    all_sheets[sheet] = df

# Save back to Excel, preserving column widths
output_path = "/content/drive/MyDrive/ISI/ISIclean/ISI2018drafty.xlsx"
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, sheet_df in all_sheets.items():
        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
        # Retain column widths
        workbook = writer.book
        worksheet = writer.sheets[sheet_name]
        for col_num, column_cells in enumerate(sheet_df.columns, start=1):
            max_length = max(sheet_df[column_cells].astype(str).map(len).max(), len(column_cells))
            worksheet.column_dimensions[openpyxl.utils.get_column_letter(col_num)].width = max_length

print(f"Processing complete. Updated file with 'Affiliation Location' column saved as {output_path}.")


Processing sheet: Authorwise 2018
Processing sheet: Chair of regular sessions
Processing sheet: SPECIAL sessions Keynote, Plena
Processing complete. Updated file with 'Affiliation Location' column saved as /content/drive/MyDrive/ISI/ISIclean/ISI2018drafty.xlsx.


Add Gender

In [13]:
!pip install requests
!pip install gender-guesser
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 379.3/379.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 3.6 MB/s eta 0:00:00


In [16]:
import pandas as pd
import gender_guesser.detector as gender
from tqdm import tqdm
import openpyxl

# Function to get gender using gender-guesser
def get_gender(name):
    if not isinstance(name, str):  # Check if the name is a string
        return 'unknown'

    first_name = name.split()[0]  # Get first name
    d = gender.Detector()
    gender_guess = d.get_gender(first_name)

    # Convert the gender guess to a more readable format
    if gender_guess in ['male', 'mostly_male']:
        return 'Male'
    elif gender_guess in ['female', 'mostly_female']:
        return 'Female'
    else:
        return 'unknown'

# Load the Excel file
file_path = "/content/drive/MyDrive/ISI/ISIclean/ISI2018drafty.xlsx"  # Replace with your actual file path
xls = pd.ExcelFile(file_path)

# Dictionary to store DataFrames
dfs = {sheet_name: pd.read_excel(xls, sheet_name=sheet_name) for sheet_name in xls.sheet_names}

# Correct column names in each sheet
sheet_columns = {
    'Authorwise 2018': 'Author',
    'SPECIAL sessions Keynote, Plena': 'Presenter/Panelist/Moderator',  # Correct column name
    'Chair of regular sessions': 'Name of Chair'  # Correct column name
}

# Print out sheet names and columns to debug
print("Sheets in the file:", xls.sheet_names)

# Debugging: Print columns for each sheet
for sheet_name in dfs.keys():
    print(f"Columns in sheet '{sheet_name}': {dfs[sheet_name].columns.tolist()}")

# Process the sheets that need gender detection
for sheet_name, column_name in sheet_columns.items():
    if sheet_name in dfs:  # Ensure the sheet exists
        if column_name in dfs[sheet_name].columns:  # Ensure the column exists
            tqdm.pandas(desc=f"Processing {sheet_name}")
            dfs[sheet_name]['Gender'] = dfs[sheet_name][column_name].progress_apply(get_gender)
        else:
            print(f"Column '{column_name}' not found in sheet '{sheet_name}'")
    else:
        print(f"Sheet '{sheet_name}' not found in the Excel file")

# Save back to Excel, preserving column widths
output_file_path = "/content/drive/MyDrive/ISI/ISIclean/ISI2018drafty.xlsx"
with pd.ExcelWriter(output_file_path, engine="openpyxl") as writer:
    for sheet_name, sheet_df in dfs.items():
        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)

        # Retain column widths
        workbook = writer.book
        worksheet = writer.sheets[sheet_name]
        for col_num, column_cells in enumerate(sheet_df.columns, start=1):
            max_length = max(sheet_df[column_cells].astype(str).map(len).max(), len(column_cells))
            worksheet.column_dimensions[openpyxl.utils.get_column_letter(col_num)].width = max_length

print(f"Processing complete. Updated file saved as {output_file_path}.")


Sheets in the file: ['Authorwise 2018', 'Chair of regular sessions', 'SPECIAL sessions Keynote, Plena']
Columns in sheet 'Authorwise 2018': ['Session', 'Topic', 'Title of Paper', 'Number of Authors', 'Author', 'Institute/Affiliation', 'Affiliation Country', 'Affiliation Location']
Columns in sheet 'Chair of regular sessions': ['Conference Year', 'Name of Chair', 'Affiliation', 'Session Theme', 'Affiliation Country', 'Affiliation Location']
Columns in sheet 'SPECIAL sessions Keynote, Plena': ['Session Type', 'Session Number', 'Presenter/Panelist/Moderator', 'Affiliation', 'Topic', 'Affiliation Country', 'Affiliation Location']


Processing Chair of regular sessions: 100%|██████████| 50/50 [00:18<00:00,  2.68it/s]

Processing complete. Updated file saved as /content/drive/MyDrive/ISI/ISIclean/ISI2018drafty.xlsx.


Rename columns and save

In [19]:
import pandas as pd

# Load the Excel file
file_path = "/content/drive/MyDrive/ISI/ISIclean/ISI2018drafty.xlsx"  # Replace with your actual file path
xls = pd.ExcelFile(file_path)

# Dictionary to store DataFrames
dfs = {sheet_name: pd.read_excel(xls, sheet_name=sheet_name) for sheet_name in xls.sheet_names}

# Loop through each sheet and rename the column
for sheet_name, sheet_df in dfs.items():
    if 'Affiliation Location' in sheet_df.columns:  # Check if the column exists in the sheet
        sheet_df.rename(columns={'Affiliation Location': 'Affiliation Location India'}, inplace=True)

# Save the updated file back to Excel
output_file_path = "/content/drive/MyDrive/ISI/ISIclean/ISI2018drafty.xlsx"
with pd.ExcelWriter(output_file_path, engine="openpyxl") as writer:
    for sheet_name, sheet_df in dfs.items():
        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)

        # Retain column widths
        workbook = writer.book
        worksheet = writer.sheets[sheet_name]
        for col_num, column_cells in enumerate(sheet_df.columns, start=1):
            max_length = max(sheet_df[column_cells].astype(str).map(len).max(), len(column_cells))
            worksheet.column_dimensions[openpyxl.utils.get_column_letter(col_num)].width = max_length

print(f"Processing complete. Updated file saved as {output_file_path}.")


Processing complete. Updated file saved as /content/drive/MyDrive/ISI/ISIclean/ISI2018drafty.xlsx.


# Experimental

In [ ]:
import spacy
import re

# Load the English NLP model
nlp = spacy.load("en_core_web_sm")

# Function to process text with NLP and add markers
def add_markers_with_nlp(text):
    doc = nlp(text)
    updated_text = []
    delimiter = "***NEW PAPER***"

    session_re = re.compile(r'(Session \d+[a-z]* [^\n]* \([^\)]*\)\n)')
    session_blocks = session_re.split(text)

    for i, block in enumerate(session_blocks):
        if i % 2 == 0:
            updated_text.append(block)
        else:
            papers = re.split(r'(?:"[^"]+"\s*\n)', block)  # Split block at paper titles
            for j, paper in enumerate(papers):
                if j == 0:
                    updated_text.append(paper)
                else:
                    updated_text.append(f'{delimiter} "{papers[j-1].split(": ")[-1].strip()}" {paper}')

    return ''.join(updated_text)

# Assuming 'text' is already loaded
updated_text = add_markers_with_nlp(text)

# Print the updated text
print(updated_text)


Parallel Sessions 1                                                18 December, Monday 11:50 AM- 01:00  PM    
Session 1(a) Social Identity and Financial Inclusion (Conference Room) 
Rikhia Bhukta (Indian Institute of Technology Kanpur), Debayan Pakrashi (Indian Statistical Institute Kolkata; 
Indian Institute of Technology Kanpur), Sarani Saha (Indian Institute of Technology Kanpur), and Ashish Sedai 
(University of Texas at Arlington): "Does Financial Inclusion Mitigate Social Exclusion? Causal Evidence from 
India" 

Sandhya Garg (Institute of Economic Growth), Samarth Gupta (Ahmedabad University), and Sushanta Mallick 
(Queen Mary University of London): "Does Social Identity Constrain Rural Entrepreneurship? The Role of 
Financial Inclusion" 
Session 1(b) Welfare Programmes I (Seminar Room 1) 
Vidhya Unnikrishnan (University of Manchester): "Impact of Social Protection Policy Targeted on the 
Disabled: Panel Data Evidence from India" 


Michael Palmer (University of Western Austral

In [ ]:
import spacy
import pandas as pd
import re

# Load the English NLP model
nlp = spacy.load("en_core_web_sm")

# Define a function to extract session details using NLP
def extract_session_details(text):
    doc = nlp(text)

    sessions = []

    # Regular expression to identify sessions and topics
    session_re = re.compile(r"Session (\d+[a-z]*)([^\n]*)\s*\(([^)]*)\)")

    # Regular expression to match the authors and affiliations
    author_re = re.compile(r"([A-Za-z\s]+)\s?\(([^)]+)\)")

    # Regular expression to capture paper titles
    title_re = re.compile(r'"\*\*\*NEW PAPER\*\*\* ([^\"]+)"')

    # Iterate over all sessions in the text
    for match in session_re.finditer(text):
        session_number, session_topic, location = match.groups()

        # Locate the paper(s) associated with the session
        session_start = match.end()
        session_end = text.find("Session", session_start)  # Find next session to stop the current one
        session_content = text[session_start:session_end] if session_end != -1 else text[session_start:]

        # Extract the paper titles
        paper_titles = title_re.findall(session_content)

        # Extract authors and their affiliations
        authors_and_affiliations = author_re.findall(session_content)

        # Process each paper
        for paper_title in paper_titles:
            authors = []
            affiliations = []

            for author_name, affiliation in authors_and_affiliations:
                authors.append(author_name.strip())
                affiliations.append(affiliation.strip())

            # Loop over authors to create a row for each
            for author, affiliation in zip(authors, affiliations):
                sessions.append({
                    "Session": session_number,
                    "Topic": session_topic,
                    "Title of Paper": paper_title,
                    "Number of Authors": len(authors),
                    "Author": author,
                    "Institute/Affiliation": affiliation
                })

    # Convert the session details into a DataFrame
    df = pd.DataFrame(sessions)

    return df

# Extract session details from the conference text
conference_df = extract_session_details(updated_text)

# Display the structured DataFrame
print(conference_df)

# Save to CSV
conference_df.to_csv("/content/drive/My Drive/conference_details.csv", index=False)


Empty DataFrame
Columns: []
Index: []
